# Quantum Oracle Sketching

> **Quantum Oracle Sketching** is a data-loading algorithm introduced by Zhao et al. [[1]](#ref1). It provides the missing link between classical data access and the *coherent oracle queries* on which many powerful quantum algorithms are built.
>
> The algorithm approximates the desired oracle $O_f$ on the fly from a stream of classical samples $(x_t, f(x_t))$, applying an incremental data-dependent rotation per sample and discarding each sample once consumed — at no point is the dataset stored in either classical or quantum memory. The result is an exponential advantage in space complexity over any classical learner, and — when the data distribution varies in time while the learner stays fixed — a super-polynomial advantage in sample complexity as well: a classical machine requires the number of data examples to scale super-polynomially with the data size $N$, while a quantum learner only needs an amount of data that scales linearly, achieving matching accuracy and probabilistic performance.
>
> The algorithm treats the following problem:
>
> - **Input:** $M$ classical data samples $\{(x_t, f(x_t))\}_{t=1}^{M}$, where each $x_t \in [N] = \{0,1,\dots,N-1\}$ is drawn independently from a (possibly time-varying) distribution $p$, and $f$ is the target function — Boolean ($f:[N]\to\{0,1\}$), real-valued (a vector $f:[N]\to\mathbb{R}$), or matrix-valued ($f:[N]\times[N]\to\mathbb{R}$).
> - **Output:** A quantum unitary $V$ on $n = \lceil\log_2 N\rceil$ qubits that approximates the corresponding *coherent* query oracle $O$ — the phase oracle $|x\rangle \to (-1)^{f(x)}|x\rangle$, the state-preparation unitary $|0\rangle \to \sum_x f(x)|x\rangle/\|f\|$, or a block encoding of the matrix — to error $\epsilon$ in diamond distance. $V$ can then be used as a drop-in subroutine inside any quantum query algorithm.
>
> ---
>
> **Complexity.**
>
> - **Sample complexity.** A single $\epsilon$-accurate oracle sketch requires $M = \Theta(N/\epsilon)$ classical samples. To support an arbitrary quantum algorithm that makes $Q$ queries to total error $\epsilon$, each query is sketched to error $\epsilon/Q$, giving
> $$ M_\text{total} = \Theta\!\bigl(N Q^2 / \epsilon\bigr). $$
> The quadratic dependence on $Q$ is unavoidable, mirroring the Born-rule relationship between quantum amplitudes and probabilities.
> - **Space complexity.** $\mathcal{O}(\mathrm{poly}\log N)$ qubits — the quantum machine never stores the dataset, only its current quantum state.
> - **Time complexity.** $\widetilde{\mathcal{O}}(N)$ in the data-loading stage (one constant-depth controlled gate per sample); subsequent processing of each sample requires only $\mathrm{poly}\log N$ time.
> - **Classical lower bound (dynamic case).** Any classical machine that matches the quantum prediction accuracy on a time-varying distribution must either use memory $\Omega(N^{0.99})$ or consume super-polynomially many samples (Theorem 7 of [[1]](#ref1)). This is the source of the exponential space advantage.
>
> ---
>
> **Keywords:** Quantum Learning Theory, Quantum Machine Learning (QML)

## Introduction

Almost every quantum speedup over a classical baseline is stated relative to an *oracle* — a unitary $O_f$ that encodes the input function $f$ and acts in superposition. [Grover's search](https://en.wikipedia.org/wiki/Grover%27s_algorithm) assumes a phase oracle $O_f|x\rangle = (-1)^{f(x)}|x\rangle$; [Shor's period-finding](https://en.wikipedia.org/wiki/Shor%27s_algorithm) queries a modular-exponentiation oracle; the [HHL linear-system solver](https://en.wikipedia.org/wiki/Quantum_algorithm_for_linear_systems_of_equations) [[5]](#ref5), [quantum singular-value transformation (QSVT)](https://arxiv.org/abs/1806.01838) [[4]](#ref4), and most quantum machine-learning routines rely on a [block-encoding oracle](https://arxiv.org/abs/1806.01838) for the input matrix; [amplitude estimation](https://en.wikipedia.org/wiki/Amplitude_amplification) and [quantum walks](https://en.wikipedia.org/wiki/Quantum_walk) similarly assume coherent access to the input. This *coherent* access — the ability to evaluate $f$ on a superposition of inputs in a single query — is exactly what enables the quadratic and exponential speedups; without it, those algorithms collapse to their classical counterparts.

Yet real-world classical data — produced one sample at a time by experiments, sensors, or users — does not natively support such queries. The standard remedy is to load the entire dataset into a quantum random-access memory (QRAM [[3]](#ref3)), but its fault-tolerance overhead often exceeds the quantum advantage it enables, leaving the most powerful quantum algorithms without a practical interface to massive classical data.

Quantum oracle sketching is the algorithm that closes this gap: it constructs an approximate $O_f$ on the fly from a stream of classical samples, with no QRAM and no full dataset ever held in memory.

## The streaming-learning framework

Quantum oracle sketching is proposed in the context of  a specific online-learning model that the paper formalizes (Sec. II and Appendix C 1 of [[1]](#ref1)). It is worth stating explicitly, because it sets the rules under which both the quantum and the classical baselines operate, and because it explains *why* the comparison between the two is well-posed.

**Setup.**

- A *data-generating process* $\mathcal{D}$ emits a sequence of classical samples $z_1, z_2, \dots, z_M$. Each sample is one observation of the underlying object — a function value $z_t = (x_t, f(x_t))$ for a Boolean target, a feature-vector / label pair $z_t = (i_t, \vec{x}_{i_t}, y_{i_t})$ for classification, or an entry $z_t = (i_t, j_t, A_{i_t j_t}, k_t, b_{k_t})$ of a linear system $A\vec{x} = \vec{b}$.
- A *learner* of size $S$ (= number of logical qubits for a quantum learner, = number of floating-point words for a classical one) observes the samples *one at a time*. After seeing $z_t$ it updates its memory, then discards $z_t$ — at no point is the dataset stored in full. This is the standard *streaming* or *online-learning* model.
- After $M$ samples, the learner is asked to predict, classify, or compress some property of the underlying process $\mathcal{D}$ — not of the empirical sample. Examples: the label of a held-out test feature vector (classification), the value of a quadratic form $\vec{x}^\top \mathcal{M} \vec{x}$ (linear systems), or the projection of a test point onto the top principal direction (PCA).

The two resources of interest are the *machine size* $S$ and the *sample count* $M$. The quantum learner aims to minimize both simultaneously.

**Static versus dynamic data.** The framework allows the data distribution to drift in time, characterized by two parameters:

- *Refreshing time* $\tau$: the timescale beyond which samples become effectively uncorrelated.
- *Repetition number* $R$: the maximum expected number of times any single sample is repeated within a window of $\tau$ steps.

When $\tau \to \infty$ we recover the static, i.i.d. setting (Theorems 1, 3, 5 of [[1]](#ref1)). When $\tau = \widetilde{\mathcal{O}}(N)$ the distribution refreshes on the same timescale as the data size, and the quantum advantage in sample complexity becomes *super-polynomial* — this is the regime of Theorems 2, 4, 6 of [[1]](#ref1), where any $\Omega(N^{0.99})$-size classical machine needs super-polynomially many samples while $\mathrm{poly}\log N$ qubits suffice with $\widetilde{\mathcal{O}}(N)$ samples.

**End-to-end pipeline.** The full quantum learner is a three-stage assembly:

1. **Quantum oracle sketching** consumes the stream of classical samples and produces an approximate oracle unitary $V \approx O_f$ (the subject of *this* notebook).
2. A **quantum query algorithm** — Grover, QSVT [[4]](#ref4), HHL [[5]](#ref5), amplitude estimation, etc. — uses $V$ as a black-box subroutine to prepare a target state $|\psi_\text{target}\rangle$ that encodes the desired property of $\mathcal{D}$.
3. **Interferometric classical shadows** (Theorem F.16 of [[1]](#ref1), a sign-preserving variant of the Huang–Kueng–Preskill protocol [[2]](#ref2)) extract a compact classical model from $|\psi_\text{target}\rangle$ in $\mathrm{poly}\log N$ measurements, ready for offline prediction on arbitrary test inputs.

This notebook covers stage 1 in detail: the Boolean phase oracle in full (numpy reference, scaling analysis, and a Classiq circuit), the real-valued state sketch through a Hadamard-test conversion (numpy), and the streaming runtime view that motivates the whole framework. Sparse-matrix block encodings are sketched at the level of theory only and left to a follow-up notebook, together with the QSVT-based stage 2 and the interferometric-shadow readout of stage 3.

## Algorithm

We aim to implement the Boolean phase oracle

$$
O\,|x\rangle = (-1)^{f(x)}\,|x\rangle, \qquad x \in [N],\ f : [N] \to \{0,1\},
$$

given only random classical samples $z_t = (x_t,\, f(x_t))$ for $t = 1,\dots,M$, where each $x_t$ is drawn uniformly from $[N] = \{0,1,\dots,N-1\}$.

**Per-sample rotation.** For each sample we apply a small data-dependent phase rotation that fires only on the basis state $|x_t\rangle$:

$$
V_t \;=\; \exp\!\bigl(i\,\tau\, f(x_t)\,|x_t\rangle\!\langle x_t| / M\bigr). \tag{1}
$$

**Coherent accumulation.** Because every $V_t$ is diagonal in the computational basis, all factors commute and the product collapses cleanly:

$$
V \;\equiv\; \prod_{t=1}^M V_t \;=\; \exp\!\Bigl(i\tfrac{\tau}{M}\sum_{t=1}^M f(x_t)\,|x_t\rangle\!\langle x_t|\Bigr) \;=\; \sum_{x=0}^{N-1} \exp\bigl(i\,\tau\, m_x\, f(x)\bigr)\,|x\rangle\!\langle x|, \tag{2}
$$

where $m_x = \frac{1}{M}\sum_t \mathbf{1}[x_t = x]$ is the empirical frequency of basis label $x$.

**Concentration.** As $M$ grows, $m_x \to p(x) = 1/N$ by the law of large numbers. Picking $\tau = \pi N$ makes $\tau\, m_x \to \pi$ and

$$
V \;\longrightarrow\; \sum_{x=0}^{N-1} e^{i\pi f(x)}\,|x\rangle\!\langle x| \;=\; \sum_x (-1)^{f(x)}\,|x\rangle\!\langle x| \;=\; O. \tag{3}
$$

**Sample complexity.** Theorem D.12 of [[1]](#ref1) shows the diamond-distance error decays as $\epsilon = \mathcal{O}(N/M)$, so reaching error $\epsilon$ costs

$$
M = \Theta(N/\epsilon) \tag{4}
$$

samples. Each sample is processed once and immediately discarded.

**Comparison to generic random Hamiltonian simulation.** The $N/M$ scaling above is *not* what a naive random-rotation strategy would deliver. If one instead applied a sequence of small phases $\exp(i\,\tau\, h_{z_t}/M)$ driven by an arbitrary Hamiltonian $h_z$ with $\|h_z\| = \mathcal{O}(1)$ — hoping the sequence approximates $\exp(i\,\tau\,\mathbb{E}[h_z])$ — randomized-Hamiltonian-simulation results [[1]](#ref1) (Sec. D 2) show the operator-norm error compounds to

$$
\epsilon_\text{naive} \;=\; \mathcal{O}(N^2 / M),\qquad \text{i.e.}\qquad M_\text{naive} \;=\; \Theta(N^2 / \epsilon).
$$

That extra factor of $N$ would consume the entire quantum advantage. To support $Q$ queries with combined error $\epsilon$, each individual query must be sketched to error $\epsilon/Q$, so per-query naive simulation needs $\Theta(N^2 Q / \epsilon)$ samples and the total becomes

$$
M_\text{total}^\text{naive} \;=\; Q \cdot \Theta(N^2 Q/\epsilon) \;=\; \Theta(N^2 Q^2/\epsilon),
$$

versus the $\Theta(NQ^2/\epsilon)$ achieved here — a factor-$N$ blow-up that would take $\widetilde{\mathcal{O}}(N)$-sample tasks straight back into the regime where any classical machine of size $\Omega(N^{0.99})$ wins. The improvement to $\epsilon = \mathcal{O}(N/M)$ comes from a crucial piece of structure: each $V_t$ phases only the *one-dimensional* subspace $|x_t\rangle\!\langle x_t|$, and distinct values of $x$ label *mutually orthogonal* subspaces, so per-sample errors live in disjoint Hilbert-space blocks rather than compounding across the full register. Quantum oracle sketching is, in this sense, a carefully engineered *non-generic* instance of randomized Hamiltonian simulation — designed precisely to dodge the $N^2/M$ trap.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from classiq import *

from oracle_sketching import (
    apply_basis_phase,
    ideal_oracle,
    quantum_oracle_sketch,
    real_valued_oracle_sketch,
    sketched_oracle,
    state_sketch,
    state_sketch_circuit,
    stream_sketch,
)

np.random.seed(7)

In [ ]:
NUM_QUBITS = 3
N = 2**NUM_QUBITS

# A random Boolean target function f: [N] -> {0, 1}
F_TABLE = np.random.randint(0, 2, size=N)
print(f"f(x) for x = 0,...,{N-1}:  {F_TABLE.tolist()}")

### Reference implementation (numpy)

We first build the sketched unitary $V$ classically by collecting $M$ random samples and assembling the diagonal matrix from eq. (2). This is the ground truth that the quantum circuit will reproduce.

In [ ]:
M = 4000
x_samples = np.random.randint(0, N, size=M)

V = sketched_oracle(F_TABLE, x_samples)
O = ideal_oracle(F_TABLE)

err = np.linalg.norm(V - O, ord=2)
print(f"Spectral-norm error  ||V - O||  for M={M}, N={N}:  {err:.3e}")
print(f"Predicted scaling    N / M                     :  {N/M:.3e}")

### Sample complexity

There are two scalings to keep distinct:

- **Single-realization** operator-norm error $\|V_\lambda - O\|_2$ for one random data sample $\lambda = (x_1, \dots, x_M)$ is dominated by the *variance* of the empirical frequencies, $m_x - 1/N = \mathcal{O}(1/\sqrt{NM})$, giving $\|V_\lambda - O\|_2 = \mathcal{O}(N/\sqrt{M})$.

- **Random-unitary channel** $\mathcal{C}(\rho) = \mathbb{E}_\lambda[V_\lambda\, \rho\, V_\lambda^\dagger]$ — the actual object that gets used inside any quantum query algorithm — has diamond-distance error $\mathcal{O}(N/M)$ to $O\rho O^\dagger$ (Theorem D.12 of [[1]](#ref1)). The improvement comes because the random unitary is unbiased to leading order: $\|\mathbb{E}[V_\lambda] - O\| = \mathcal{O}(N/M)$, an order of magnitude better than a single realization.

This second scaling is the one that drives the $M = \Theta(N/\epsilon)$ sample complexity. We verify both empirically below by sweeping $M$ over three decades and (for the channel) averaging $V_\lambda$ over independent runs.

In [ ]:
M_values = np.unique(np.logspace(2, 4.5, 12, dtype=int))
n_trials = 200

single_err = np.empty(M_values.size)
channel_err = np.empty(M_values.size)
for k, M_k in enumerate(M_values):
    Vs = [
        sketched_oracle(F_TABLE, np.random.randint(0, N, size=int(M_k)))
        for _ in range(n_trials)
    ]
    single_err[k] = np.mean([np.linalg.norm(V - O, ord=2) for V in Vs])
    channel_err[k] = np.linalg.norm(np.mean(Vs, axis=0) - O, ord=2)

c_single = float(np.median(single_err * np.sqrt(M_values) / N))
c_channel = float(np.median(channel_err * M_values / N))

fig, ax = plt.subplots(figsize=(6, 4.5))
ax.loglog(M_values, single_err, "o-", label=r"single realization $\|V_\lambda - O\|_2$")
ax.loglog(M_values, channel_err, "s-", label=r"channel bias $\|\mathbb{E}[V_\lambda] - O\|_2$")
ax.loglog(M_values, c_single * N / np.sqrt(M_values), "--", color="C0", alpha=0.5, label=fr"${c_single:.2f}\,N/\sqrt{{M}}$")
ax.loglog(M_values, c_channel * N / M_values, "--", color="C1", alpha=0.5, label=fr"${c_channel:.2f}\,N/M$")
ax.set_xlabel("Number of samples $M$")
ax.set_ylabel("Spectral-norm error")
ax.set_title(f"Boolean oracle sketching, $N = {N}$, {n_trials} trials")
ax.legend(fontsize=9)
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

### Quantum circuit (Classiq)

Each sample contributes a single factor $V_t$. When $f(x_t)=0$ the factor is the identity and the sample is dropped; when $f(x_t)=1$ we apply $\exp(i\theta\,|x_t\rangle\!\langle x_t|)$ with the fixed angle $\theta = \tau / M = \pi N / M$.

The primitive `apply_basis_phase` realises $\exp(i\theta\,|x_\text{int}\rangle\!\langle x_\text{int}|)$ by controlling on the equality predicate `qvar == x_int`; the body `phase(theta)` becomes a relative phase on the controlled subspace and the identity elsewhere — exactly $V_t$.

We pre-filter the classical data so only samples with $f(x_t)=1$ enter the circuit (the rest contribute the identity), fix $\theta = \pi N / M$, and bake the samples into the circuit at synthesis time. The leading Hadamard transform lets us see the diagonal-phase action of $V$ on the uniform superposition $|+\rangle^{\otimes n}$.

In [ ]:
# For circuit synthesis we use a smaller M so the visualization stays readable;
# the numpy benchmark above already validates the algorithm at larger M.
M_demo = 200
demo_samples = np.random.randint(0, N, size=M_demo)
positive_samples = demo_samples[F_TABLE[demo_samples] == 1].tolist()
theta = np.pi * N / M_demo  # τ / M with τ = π N — divide by total M, not by len(positive_samples)


@qfunc
def main(qvar: Output[QNum]) -> None:
    allocate(NUM_QUBITS, qvar)
    hadamard_transform(qvar)
    quantum_oracle_sketch(theta, positive_samples, qvar)


qprog = synthesize(main)
show(qprog)

## Extensions

The Boolean phase oracle is the cleanest setting in which to see the algorithm at work, but the same primitive — an incremental, basis-state-targeted phase rotation — extends to all of the standard quantum-input formats. The constructions sketched in Sec. D.4–D.6 of [[1]](#ref1) layer two ideas on top of what we have already built:

1. **Per-sample real-valued angles.** Replacing the Boolean $f(x_t) \in \{0,1\}$ with a continuous value $v_{x_t} \in [-1, 1]$ and rescaling $\tau$ accordingly turns the sketch into a phase oracle for any real-valued function — without changing the gate structure.
2. **Hadamard-test conversion plus oblivious amplitude amplification.** A single ancilla qubit converts the phase oracle into a block encoding of a diagonal matrix; *oblivious amplitude amplification* (OAA) then raises that block to unit amplitude, yielding a deterministic state-preparation unitary.

Below we walk through both extensions and close with the streaming runtime view that motivated the algorithm in the first place.

### State sketching for real-valued vectors

**Goal.** Given a real unit vector $\vec{v}\in\mathbb{R}^N$ with $|v_x| \le 1$, prepare $|\vec{v}\rangle = \sum_x v_x|x\rangle$ from classical samples $(x_t,\,v_{x_t})$ with $x_t \sim \mathrm{Unif}([N])$.

**Sketched phase oracle.** Generalising the per-sample gate to admit a real-valued angle,

$$
V_t \;=\; \exp\!\bigl(i\,\tau\, v_{x_t}\, |x_t\rangle\!\langle x_t| / M\bigr),
$$

the same product-and-concentration argument gives

$$
V \;\longrightarrow\; \sum_x e^{i \tau\, p(x)\, v_x}\,|x\rangle\!\langle x|.
$$

Choosing $\tau = \theta N$ with a small *amplitude angle* $\theta \ll 1$ produces the phase oracle $V|x\rangle = e^{i\theta v_x}\,|x\rangle$.

**Hadamard-test conversion.** A single ancilla qubit converts the phase into an amplitude. Starting from $|0\rangle_a \otimes |+\rangle^{\otimes n}$ and applying $H_a \cdot \mathrm{C\text{-}}V \cdot H_a$,

$$
|0\rangle_a |+\rangle^{\otimes n} \;\longmapsto\; \frac{1}{\sqrt{N}}\sum_x \Bigl(\cos\!\tfrac{\theta v_x}{2}\,|0\rangle_a + i\,\sin\!\tfrac{\theta v_x}{2}\,|1\rangle_a\Bigr) |x\rangle.
$$

Post-selecting on $|1\rangle_a$ leaves the state $\propto \sum_x \sin(\theta v_x/2)\,|x\rangle \approx (\theta/2)\,|\vec v\rangle$ for $\theta \to 0$. The post-selection success probability is $p_\text{succ} = \Theta(\theta^2)$.

**Oblivious amplitude amplification (OAA).** Applying OAA $\mathcal{O}(1/\theta)$ times raises the post-selected branch to unit amplitude, yielding a *deterministic* state-preparation unitary $U_v$ with $U_v|0\rangle = |\vec v\rangle$ at the cost of $\mathcal{O}(1/\theta)$ additional sketch invocations. We omit the OAA inner loop in the demo below — it is a fixed sequence of reflections, mechanical to implement — and instead verify the *post-selected* amplitude vector against the target.

In [ ]:
# Toy target: a random real unit vector
NUM_QUBITS_V = 3
N_V = 2**NUM_QUBITS_V
np.random.seed(11)
v_target = np.random.randn(N_V)
v_target /= np.linalg.norm(v_target)

# Sketch
amp_angle = np.pi / 16
M_v = 5000
x_samples_v = np.random.randint(0, N_V, size=M_v)
amps = state_sketch(v_target, x_samples_v, amp_angle)

# In the small-θ limit, amps ≈ (θ/2) v_x / sqrt(N).
# Recover the estimated state and check its overlap with the target.
v_est = amps / (amp_angle / 2) * np.sqrt(N_V)
v_est /= np.linalg.norm(v_est)
print(f"target |v⟩ :  {np.round(v_target, 3)}")
print(f"v_est      :  {np.round(v_est, 3)}")
print(f"|⟨v_est | v⟩| = {abs(v_est @ v_target):.4f}    (1 = perfect)")

In [ ]:
M_values_v = np.unique(np.logspace(2, 4.5, 12, dtype=int))
n_trials_v = 100

infidelity = np.empty(M_values_v.size)
for k, M_k in enumerate(M_values_v):
    fids = []
    for _ in range(n_trials_v):
        xs = np.random.randint(0, N_V, size=int(M_k))
        a = state_sketch(v_target, xs, amp_angle)
        v_est_k = a / (amp_angle / 2) * np.sqrt(N_V)
        v_est_k /= np.linalg.norm(v_est_k) + 1e-15
        fids.append(abs(v_est_k @ v_target))
    infidelity[k] = 1 - np.mean(fids)

c_fit = float(np.median(infidelity * M_values_v / N_V))

fig, ax = plt.subplots(figsize=(5, 4))
ax.loglog(M_values_v, infidelity, "o-", label=r"$1 - |\langle v_\mathrm{est}\, |\, v\rangle|$")
ax.loglog(M_values_v, c_fit * N_V / M_values_v, "--", label=fr"${c_fit:.2f}\,N/M$ guide")
ax.set_xlabel("Number of samples $M$")
ax.set_ylabel("Infidelity")
ax.set_title(f"State sketching, $N={N_V}$, $\\theta=\\pi/16$, {n_trials_v} trials")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

**Classiq circuit for state sketching.** The numpy reference above operates directly on diagonal phases. Below we build the same construction as a Classiq `@qfunc` so the post-selected branch becomes an actual quantum state-preparation routine. The circuit has three pieces:

1. `real_valued_oracle_sketch` — a real-valued generalisation of `quantum_oracle_sketch` where each sample $t$ carries its *own* rotation angle (the only way a single fixed $\theta$ is no longer enough is precisely the move from Boolean $f$ to real-valued $\vec v$).
2. `state_sketch_circuit` — the Hadamard-test wrapper: `H(aux); control(aux == 1, real-valued sketch); H(aux)`. Conditioning the entire sketched phase oracle on `aux == 1` is what converts the diagonal phase $e^{i\theta v_x}$ into amplitudes $\sin(\theta v_x/2)$ on the $|1\rangle_a$ branch.
3. `state_sketch_main` — bakes $M = 200$ classical samples at $\theta = \pi/16$ into the circuit (the "compile-time" view from the Boolean section), allocates the registers, applies a Hadamard transform to set up the uniform superposition, and then runs the sketch. Post-selecting `aux == 1` recovers the unnormalised target amplitudes $\sin(\theta v_x/2)\,|x\rangle \approx (\theta/2)\,|\vec v\rangle$.

Synthesis is rendered for inspection — full QSVT-style amplitude amplification (which would lift the post-selected branch to unit amplitude) lives in the LS-SVM / PCA notebooks of this series.

In [ ]:
# Bake angles_per_sample[t] = θ N v_{x_t} / M so the inner product τ m_x v_x
# converges to θ v_x.
M_state_demo = 200
samples_state_demo = np.random.randint(0, N_V, size=M_state_demo)
angles_demo = (
    amp_angle * N_V / M_state_demo * v_target[samples_state_demo]
).tolist()


# Classiq requires the entry-point qfunc to be named `main`; redefining it
# here rebinds the symbol from the Boolean-oracle section above, which is
# fine — that synthesis already ran and produced its own `qprog`.
@qfunc
def main(aux: Output[QBit], qvar: Output[QNum]) -> None:
    allocate(1, aux)
    allocate(NUM_QUBITS_V, qvar)
    hadamard_transform(qvar)
    state_sketch_circuit(angles_demo, samples_state_demo.tolist(), aux, qvar)


qprog_state = synthesize(main)
show(qprog_state)

### Block encodings of sparse matrices

State sketching extracts a *vector* from classical data. The next step — extracting a *matrix* — relies on the same primitive but with two parallel oracles: a *matrix-element* oracle that returns $A_{ij}$ given the index pair $(i, j)$, and a *matrix-index* oracle that returns the column index of the $k$-th non-zero entry in row $i$. Both are sketched using the per-sample phase rotation we have already built, and both are then combined into a *block encoding* $U_A$ on $a + n$ qubits satisfying

$$
\bigl(\langle 0|^{\otimes a} \otimes I_n\bigr)\, U_A\, \bigl(|0\rangle^{\otimes a} \otimes I_n\bigr) \;=\; A / \alpha,
$$

for some normalisation $\alpha = \mathcal{O}(\|A\|_\text{max} \cdot s)$ with $s$ the row sparsity. The construction needs $\widetilde{\mathcal{O}}(N)$ classical samples per query, matching the $M = \Theta(N/\epsilon)$ scaling we proved for the Boolean case.

Once $A$ is block-encoded, the [quantum singular-value transformation](https://arxiv.org/abs/1806.01838) [[4]](#ref4) yields a polynomial of $A$ — including $A^{-1}$ for HHL-style linear-system solving and the spectral projector for PCA. This is the path to the LS-SVM and PCA tasks of Sec. F of [[1]](#ref1) and is the subject of a future notebook in this series; the QSVT machinery is substantial enough that it deserves a dedicated exposition rather than a few cells appended here.

### Compile-time vs streaming circuits

The Classiq circuit synthesised earlier bakes all $M$ samples into a single program — convenient for benchmarking but not the runtime model the algorithm was designed for. In a true streaming deployment, samples arrive one at a time; the quantum learner applies the corresponding gate immediately and discards the sample, never holding more than its current quantum state and an $\mathcal{O}(\log N)$-size register of running statistics.

The numerical content is identical — every $V_t$ is diagonal so gate ordering is irrelevant — but the runtime profile is very different: the per-sample cost is constant, and the wall-clock time scales with the data feed rather than with a one-shot compilation of the full $M$-gate circuit. In Classiq this is naturally expressed as an `ExecutionSession` that applies one fresh `apply_basis_phase` gate per arriving sample on top of the persisted quantum state.

Below we simulate the streaming view in numpy: we accumulate empirical frequencies sample by sample and snapshot the resulting unitary $V$ at increasing sample counts. The error to the ideal oracle decays smoothly as more data arrives — without ever materialising the full sample list in memory.

In [ ]:
checkpoints = [100, 250, 500, 1000, 2500, 5000, 10000]
rng = np.random.default_rng(0)
snaps = stream_sketch(F_TABLE, max(checkpoints), checkpoints, rng=rng)

print(f"{'samples':>10s}  {'||V - O||_2':>12s}")
for t, err in snaps:
    print(f"{t:>10d}  {err:>12.3e}")

## What's next

- **End-to-end LS-SVM on classical data.** Combine the matrix block encoding above with QSVT-based matrix inversion to solve a least-squares support-vector machine using only streaming classical samples (Theorem 3 of [[1]](#ref1)).
- **Dimension reduction (PCA).** Use the same block encoding to project test points onto the top principal direction of a sparse, gapped data matrix (Theorem 5 of [[1]](#ref1)).
- **Interferometric classical shadows for readout.** Compress the resulting quantum model into a compact classical predictor using the sign-preserving shadow protocol (Theorem F.16 of [[1]](#ref1), built on the Huang–Kueng–Preskill construction [[2]](#ref2)) — the final stage of the three-stage pipeline introduced earlier in this notebook.

These will be the subjects of follow-up notebooks in this series.

## References

<a id='ref1'></a>
[[1]](#ref1) Zhao, H., Zlokapa, A., Neven, H., Babbush, R., Preskill, J., McClean, J. R., and Huang, H.-Y. Exponential quantum advantage in processing massive classical data. [arXiv:2604.07639 (2026)](https://arxiv.org/abs/2604.07639)

<a id='ref2'></a>
[[2]](#ref2) Huang, H.-Y., Kueng, R., and Preskill, J. *Predicting many properties of a quantum system from very few measurements.* Nature Physics 16, 1050–1057 (2020). https://doi.org/10.1038/s41567-020-0932-7. [arXiv](https://arxiv.org/abs/2002.08953)

<a id='ref3'></a>
[[3]](#ref3) Giovannetti, V., Lloyd, S., and Maccone, L. *Quantum random access memory.* Physical Review Letters 100, 160501 (2008). https://doi.org/10.1103/PhysRevLett.100.160501. [arXiv](https://arxiv.org/abs/0708.1879)

<a id='ref4'></a>
[[4]](#ref4) Gilyén, A., Su, Y., Low, G. H., and Wiebe, N. *Quantum singular value transformation and beyond: exponential improvements for quantum matrix arithmetics.* In Proceedings of the 51st Annual ACM SIGACT Symposium on Theory of Computing, 193–204 (2019). https://doi.org/10.1145/3313276.3316366. [arXiv](https://arxiv.org/abs/1806.01838).

<a id='ref5'></a>
[[5]](#ref5) Harrow, A. W., Hassidim, A., and Lloyd, S. *Quantum algorithm for linear systems of equations.* Physical Review Letters 103, 150502 (2009). https://doi.org/10.1103/PhysRevLett.103.150502. [arXiv](https://arxiv.org/abs/0811.3171).